In [1]:
import sys
sys.path.insert(0, r"D:\BDA\nb" if sys.platform == "win32" else "/workspace/nb")
from bda_common import *

import pandas as pd
import numpy as np
from pyspark.sql import functions as F, Window
from pyspark import StorageLevel
from pyspark.ml.feature import MinHashLSH, BucketedRandomProjectionLSH

info_mesin()

spark = spark_session("06-graph", konfig={
    "spark.cleaner.referenceTracking.cleanCheckpoints": "true",
})

DIR_CKPT = jalur("graph/_ckpt")
spark.sparkContext.setCheckpointDir(DIR_CKPT)

try:
    _jvm = spark.sparkContext._jvm
    _fs = _jvm.org.apache.hadoop.fs.FileSystem.get(
        spark.sparkContext._jsc.hadoopConfiguration())
    _p = _jvm.org.apache.hadoop.fs.Path(DIR_CKPT)
    if _fs.exists(_p):
        _fs.delete(_p, True)
        print(f"  Checkpoint sisa dihapus: {DIR_CKPT}")
    spark.sparkContext.setCheckpointDir(DIR_CKPT)
except Exception as e:
    print(f"  Checkpoint sisa tidak bisa dihapus ({type(e).__name__}); dilanjutkan.")

K = CFG["k_utama"]
fitur = spark.read.parquet(jalur(f"features/kmer_k{K}"))
fitur = fitur.withColumn(
    "galur", F.coalesce(F.nullif(F.trim(F.col("isolat")), F.lit("")), F.col("accession")))
fitur.cache()
print(f"\n  Sekuens berfitur : {fitur.count():,}")
print(f"  Galur unik       : {fitur.select('galur').distinct().count():,}")

semua_kemunculan = (spark.read.parquet(jalur("stage/sequences"))
                    .select("accession", "hash_seq", "segmen", "subtipe",
                            "negara", "wilayah", "tahun_koleksi", "inang",
                            "isolat", "wakil_unik"))
semua_kemunculan = semua_kemunculan.withColumn(
    "galur", F.coalesce(F.nullif(F.trim(F.col("isolat")), F.lit("")),
                        F.col("accession")))
semua_kemunculan.cache()
n_kemunculan = semua_kemunculan.count()
print(f"  Kemunculan seluruhnya : {n_kemunculan:,}  "
      f"(termasuk {n_kemunculan - fitur.count():,} salinan identik)")
print(f"  Negara terwakili      : "
      f"{semua_kemunculan.select('negara').distinct().count():,}")


CPU logis      : 24
RAM total      : 50.5 GB   bebas 41.9 GB
Disk D: bebas  : 362.6 GB dari 1,024.1 GB
Python         : 3.10.12
Mode           : KLASTER  (hdfs://namenode:8020)
run_id         : run_20261004T162404Z


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/04 16:24:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/04 16:24:07 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


Spark 3.5.9 | master yarn | driver 8g | paralelisme 8
Spark UI: http://jupyter:4040
  Checkpoint sisa dihapus: hdfs://namenode:8020/bda/graph/_ckpt



  Sekuens berfitur : 892,344


  Galur unik       : 244,140


  Kemunculan seluruhnya : 1,586,912  (termasuk 694,568 salinan identik)
  Negara terwakili      : 172


In [2]:
def derajat(tepi):
    a = tepi.select(F.col("src").alias("id"))
    b = tepi.select(F.col("dst").alias("id"))
    return a.union(b).groupBy("id").count().withColumnRenamed("count", "derajat")

def komponen_terhubung(tepi, maks_iterasi=20):
    e = (tepi.select("src", "dst")
             .union(tepi.select(F.col("dst").alias("src"), F.col("src").alias("dst")))
             .distinct())
    e = e.checkpoint(eager=True)

    simpul = e.select(F.col("src").alias("id")).distinct()
    label = simpul.withColumn("komponen", F.col("id"))

    for it in range(maks_iterasi):
        baru = (e.join(label, e.src == label.id)
                 .groupBy(F.col("dst").alias("id"))
                 .agg(F.min("komponen").alias("kandidat")))
        gabung = (label.join(baru, "id", "left")
                       .withColumn("komponen_baru",
                                   F.least(F.col("komponen"),
                                           F.coalesce(F.col("kandidat"), F.col("komponen")))))
        label_baru = gabung.select("id", F.col("komponen_baru").alias("komponen"))

        label_baru = label_baru.checkpoint(eager=True)
        berubah = (label_baru.join(label.withColumnRenamed("komponen", "lama"), "id")
                             .filter(F.col("komponen") != F.col("lama")).count())
        label = label_baru
        print(f"      iterasi {it+1}: {berubah:,} simpul berubah label")
        if berubah == 0:
            break
    return label

def pagerank(tepi, iterasi=10, damping=0.85):

    e = tepi.select("src", "dst").checkpoint(eager=True)
    keluar = (e.groupBy("src").count()
               .withColumnRenamed("count", "derajat_keluar")
               .checkpoint(eager=True))
    simpul = (e.select(F.col("src").alias("id"))
               .union(e.select(F.col("dst").alias("id")))
               .distinct().checkpoint(eager=True))
    n = simpul.count()
    pr = simpul.withColumn("skor", F.lit(1.0 / n))
    print(f"      {n:,} simpul, {iterasi} iterasi")

    for i in range(iterasi):
        kontrib = (e.join(pr, e.src == pr.id)
                    .join(keluar, "src")
                    .select(F.col("dst").alias("id"),
                            (F.col("skor") / F.col("derajat_keluar")).alias("bagian")))
        pr = (simpul.join(kontrib.groupBy("id").agg(F.sum("bagian").alias("masuk")),
                          "id", "left")
                    .withColumn("skor", F.lit((1 - damping) / n)
                                + damping * F.coalesce(F.col("masuk"), F.lit(0.0)))
                    .select("id", "skor"))

        pr = pr.checkpoint(eager=True)
        print(f"      iterasi {i+1}/{iterasi}")
    return pr.orderBy(F.desc("skor"))


In [3]:
AMBANG_JARAK = CFG["lsh_ambang_jarak"]
SIMPUL_MAKS = CFG["lsh_simpul_maks"]
TEPI_MAKS_SEGMEN = CFG["lsh_tepi_maks"]
N_HASH = CFG["lsh_hash"]

SEG_UNTUK_GRAF = SEG_EKSTERNAL + SEG_INTERNAL
TUJUAN_TEPI = jalur("graph/tepi")

def graf_kemiripan(segmen, ambang=AMBANG_JARAK, n_hash=N_HASH):
    d = (fitur.filter(F.col("segmen") == segmen)
              .select("accession", "features"))
    n_penuh = d.count()
    if n_penuh < 2:
        return None, 0, 0

    if n_penuh > SIMPUL_MAKS:
        d = d.sample(False, SIMPUL_MAKS / n_penuh, seed=CFG["seed"])
    d = d.persist(StorageLevel.MEMORY_AND_DISK)
    n = d.count()

    lsh = MinHashLSH(inputCol="features", outputCol="hash",
                     numHashTables=n_hash, seed=CFG["seed"])
    model = lsh.fit(d)

    tepi = (model.approxSimilarityJoin(d, d, ambang, distCol="jarak")
            .filter(F.col("datasetA.accession") < F.col("datasetB.accession"))
            .select(F.col("datasetA.accession").alias("src"),
                    F.col("datasetB.accession").alias("dst"),
                    F.round(1 - F.col("jarak"), 4).alias("kemiripan"))
            .withColumn("segmen", F.lit(segmen)))

    return tepi, n_penuh, n, d

def tulis_segmen(tepi, segmen, pertama):
    (tepi.write
         .mode("overwrite" if pertama else "append")
         .partitionBy("segmen")
         .parquet(TUJUAN_TEPI))
    return (spark.read.parquet(TUJUAN_TEPI)
                 .filter(F.col("segmen") == segmen).count())

print(f"  Ambang jarak Jaccard : {AMBANG_JARAK}")
print(f"  Simpul maks/segmen   : {SIMPUL_MAKS:,}")
print(f"  Anggaran tepi/segmen : {TEPI_MAKS_SEGMEN:,}")
print(f"  Tabel hash           : {N_HASH}")
print(f"\n  Kasus terburuk teoretis: {SIMPUL_MAKS * (SIMPUL_MAKS - 1) // 2:,} "
      f"pasangan per segmen")


  Ambang jarak Jaccard : 0.12
  Simpul maks/segmen   : 2,500
  Anggaran tepi/segmen : 1,500,000
  Tabel hash           : 3

  Kasus terburuk teoretis: 3,123,750 pasangan per segmen


In [4]:
with Tahap("muat graf kemiripan yang sudah tersimpan", "GRAPH"):
    tepi = spark.read.parquet(TUJUAN_TEPI)
    print(f"  tepi kemiripan : {tepi.count():,}")

    df_tepi_ringkas = (tepi.groupBy("segmen").count()
                           .withColumnRenamed("count", "tepi")
                           .orderBy("segmen").toPandas())
    df_tepi_ringkas["gen"] = df_tepi_ringkas["segmen"].map(lambda x: SEGMEN[x][0])
    df_tepi_ringkas = df_tepi_ringkas[["segmen", "gen", "tepi"]]
    display(df_tepi_ringkas)

    print("  CATATAN: angka simpul dan durasi per segmen tidak ikut")
    print("  tersimpan, jadi ringkasan ini hanya memuat jumlah tepi.")



--------------------------------------------------------------------
[>] GRAPH | muat graf kemiripan yang sudah tersimpan


  tepi kemiripan : 358,212


,segmen,gen,tepi
0,1,PB2,60436
1,2,PB1,54814
2,3,PA,66933
3,4,HA,19233
4,5,NP,46045
5,6,NA,27749
6,7,M,45264
7,8,NS,37738


  CATATAN: angka simpul dan durasi per segmen tidak ikut
  tersimpan, jadi ringkasan ini hanya memuat jumlah tepi.
[<] OK | 2.4 detik | RAM bebas 31.9 GB


In [5]:
with Tahap("uji satu segmen sebelum menjalankan semuanya", "GRAPH"):

    SEG_UJI = 4

    t0 = time.time()
    tepi_uji, n_penuh, n_pakai, cache = graf_kemiripan(SEG_UJI)
    n_tepi = tulis_segmen(tepi_uji, SEG_UJI, pertama=True)
    cache.unpersist()
    detik = time.time() - t0

    print(f"\n  Segmen {SEG_UJI} ({SEGMEN[SEG_UJI][0]}):")
    print(f"    simpul di arsip  : {n_penuh:,}")
    print(f"    simpul dipakai   : {n_pakai:,}")
    print(f"    tepi dihasilkan  : {n_tepi:,}")
    print(f"    durasi           : {detik:,.1f} detik")
    print(f"    rata-rata tepi per simpul : {n_tepi / max(n_pakai, 1):.1f}")

    if n_tepi > TEPI_MAKS_SEGMEN:
        print(f"\n  [!] BERHENTI. {n_tepi:,} tepi melampaui anggaran "
              f"{TEPI_MAKS_SEGMEN:,}.")
        print("      Tujuh segmen sisanya TIDAK dijalankan.")
        print("      Turunkan lsh_simpul_maks atau lsh_ambang_jarak di")
        print("      bda_common.py, lalu ulangi sel ini.")
        raise RuntimeError(
            f"anggaran tepi terlampaui pada segmen uji: {n_tepi:,} "
            f"> {TEPI_MAKS_SEGMEN:,}")

    perkiraan = n_tepi * len(SEG_UNTUK_GRAF)
    print(f"\n  Lolos anggaran. Perkiraan total seluruh segmen: "
          f"~{perkiraan:,} tepi, ~{detik * len(SEG_UNTUK_GRAF) / 60:,.0f} menit.")



--------------------------------------------------------------------
[>] GRAPH | uji satu segmen sebelum menjalankan semuanya


[Stage 54:==================================================>       (7 + 1) / 8]


  Segmen 4 (HA):
    simpul di arsip  : 175,801
    simpul dipakai   : 2,604
    tepi dihasilkan  : 19,233
    durasi           : 61.9 detik
    rata-rata tepi per simpul : 7.4

  Lolos anggaran. Perkiraan total seluruh segmen: ~153,864 tepi, ~8 menit.
[<] OK | 61.9 detik | RAM bebas 29.3 GB


In [6]:
with Tahap("bangun graf kemiripan untuk segmen sisanya", "GRAPH"):
    ringkas_tepi = [{
        "segmen": SEG_UJI, "gen": SEGMEN[SEG_UJI][0],
        "simpul_arsip": n_penuh, "simpul_dipakai": n_pakai,
        "tepi": n_tepi, "detik": round(detik, 1)}]

    for s in SEG_UNTUK_GRAF:
        if s == SEG_UJI:
            continue
        t0 = time.time()
        tepi_s, np_, npk, cache = graf_kemiripan(s)
        if tepi_s is None:
            print(f"  segmen {s} ({SEGMEN[s][0]}): terlalu sedikit data -- dilewati")
            continue

        nt = tulis_segmen(tepi_s, s, pertama=False)
        cache.unpersist()
        dt = round(time.time() - t0, 1)

        ringkas_tepi.append({
            "segmen": s, "gen": SEGMEN[s][0],
            "simpul_arsip": np_, "simpul_dipakai": npk,
            "tepi": nt, "detik": dt})

        catatan = "" if np_ == npk else f"  (disampel dari {np_:,})"
        print(f"  segmen {s} ({SEGMEN[s][0]:<3}): {npk:>6,} simpul  "
              f"{nt:>8,} tepi  {dt:>6,.1f}s{catatan}")

        if nt > TEPI_MAKS_SEGMEN:
            print(f"\n  [!] Segmen {s} melampaui anggaran -- dihentikan di sini.")
            break

    df_tepi_ringkas = pd.DataFrame(ringkas_tepi)
    display(df_tepi_ringkas)

    tepi = spark.read.parquet(TUJUAN_TEPI)
    total_tepi = tepi.count()
    print(f"\n  Total tepi kemiripan : {total_tepi:,}")
    print(f"  Disimpan ke {TUJUAN_TEPI}")
    print()
    print("  CATATAN UNTUK LAPORAN: graf ini dibangun atas SAMPEL, bukan")
    print(f"  seluruh arsip. Maksimum {SIMPUL_MAKS:,} simpul per segmen dari")
    print("  populasi yang jauh lebih besar. Struktur komponen terhubung dan")
    print("  jaringan reassortment tetap terbaca, tetapi angka absolutnya")
    print("  berlaku untuk sampel. Nyatakan itu terus terang.")



--------------------------------------------------------------------
[>] GRAPH | bangun graf kemiripan untuk segmen sisanya


  segmen 6 (NA ):  2,509 simpul    27,749 tepi    62.4s  (disampel dari 129,996)


  segmen 1 (PB2):  2,472 simpul    60,436 tepi   119.4s  (disampel dari 113,464)


  segmen 2 (PB1):  2,505 simpul    54,814 tepi   348.5s  (disampel dari 110,554)


  segmen 3 (PA ):  2,536 simpul    66,933 tepi   180.7s  (disampel dari 111,134)


  segmen 5 (NP ):  2,511 simpul    46,045 tepi    81.6s  (disampel dari 93,984)


  segmen 7 (M  ):  2,581 simpul    45,264 tepi   177.2s  (disampel dari 78,961)


  segmen 8 (NS ):  2,566 simpul    37,738 tepi   131.0s  (disampel dari 78,450)


,segmen,gen,simpul_arsip,simpul_dipakai,tepi,detik
0,4,HA,175801,2604,19233,61.9
1,6,NA,129996,2509,27749,62.4
2,1,PB2,113464,2472,60436,119.4
3,2,PB1,110554,2505,54814,348.5
4,3,PA,111134,2536,66933,180.7
5,5,NP,93984,2511,46045,81.6
6,7,M,78961,2581,45264,177.2
7,8,NS,78450,2566,37738,131.0



  Total tepi kemiripan : 358,212
  Disimpan ke hdfs://namenode:8020/bda/graph/tepi

  CATATAN UNTUK LAPORAN: graf ini dibangun atas SAMPEL, bukan
  seluruh arsip. Maksimum 2,500 simpul per segmen dari
  populasi yang jauh lebih besar. Struktur komponen terhubung dan
  jaringan reassortment tetap terbaca, tetapi angka absolutnya
  berlaku untuk sampel. Nyatakan itu terus terang.
[<] OK | 1,101.6 detik | RAM bebas 25.1 GB


In [7]:
with Tahap("komponen terhubung -> garis keturunan segmen", "GRAPH"):
    komponen_per_segmen = []
    for s in SEG_UNTUK_GRAF:
        t = tepi.filter(F.col("segmen") == s).select("src", "dst")

        if t.limit(1).count() == 0:
            continue
        print(f"\n  segmen {s} ({SEGMEN[s][0]}):")

        komp = komponen_terhubung(t)
        komp = komp.withColumn("segmen", F.lit(s))
        komponen_per_segmen.append(komp)
        n_k = komp.select("komponen").distinct().count()
        print(f"    {komp.count():,} sekuens -> {n_k:,} garis keturunan")

    from functools import reduce
    lineage = reduce(lambda a, b: a.union(b), komponen_per_segmen)
    lineage = lineage.withColumn(
        "lineage_id", F.concat_ws("_", F.lit("S"), F.col("segmen"), F.col("komponen")))
    lineage.write.mode("overwrite").parquet(jalur("graph/lineage"))
    print(f"\n  Total garis keturunan : {lineage.select('lineage_id').distinct().count():,}")



--------------------------------------------------------------------
[>] GRAPH | komponen terhubung -> garis keturunan segmen

  segmen 4 (HA):
      iterasi 1: 1,380 simpul berubah label
      iterasi 2: 680 simpul berubah label
      iterasi 3: 455 simpul berubah label
      iterasi 4: 285 simpul berubah label
      iterasi 5: 133 simpul berubah label
      iterasi 6: 48 simpul berubah label
      iterasi 7: 30 simpul berubah label
      iterasi 8: 51 simpul berubah label
      iterasi 9: 10 simpul berubah label
      iterasi 10: 8 simpul berubah label
      iterasi 11: 0 simpul berubah label
    1,594 sekuens -> 154 garis keturunan

  segmen 6 (NA):
      iterasi 1: 1,418 simpul berubah label
      iterasi 2: 966 simpul berubah label
      iterasi 3: 685 simpul berubah label
      iterasi 4: 562 simpul berubah label
      iterasi 5: 530 simpul berubah label
      iterasi 6: 472 simpul berubah label
      iterasi 7: 445 simpul berubah label
      iterasi 8: 264 simpul berubah label


      iterasi 6: 733 simpul berubah label
      iterasi 7: 576 simpul berubah label
      iterasi 8: 295 simpul berubah label
      iterasi 9: 7 simpul berubah label
      iterasi 10: 0 simpul berubah label
    1,803 sekuens -> 119 garis keturunan

  segmen 3 (PA):


      iterasi 1: 1,714 simpul berubah label
      iterasi 2: 1,233 simpul berubah label
      iterasi 3: 994 simpul berubah label
      iterasi 4: 874 simpul berubah label
      iterasi 5: 755 simpul berubah label
      iterasi 6: 436 simpul berubah label
      iterasi 7: 370 simpul berubah label
      iterasi 8: 268 simpul berubah label
      iterasi 9: 2 simpul berubah label
      iterasi 10: 0 simpul berubah label
    1,862 sekuens -> 124 garis keturunan

  segmen 5 (NP):
      iterasi 1: 1,574 simpul berubah label
      iterasi 2: 901 simpul berubah label
      iterasi 3: 717 simpul berubah label
      iterasi 4: 570 simpul berubah label
      iterasi 5: 394 simpul berubah label
      iterasi 6: 322 simpul berubah label
      iterasi 7: 106 simpul berubah label
      iterasi 8: 0 simpul berubah label
    1,741 sekuens -> 140 garis keturunan

  segmen 7 (M):
      iterasi 1: 1,655 simpul berubah label
      iterasi 2: 1,313 simpul berubah label
      iterasi 3: 1,047 simpul berubah 

In [8]:
with Tahap("ukuran garis keturunan", "GRAPH"):
    ukuran = (lineage.groupBy("segmen", "lineage_id").count()
                     .withColumnRenamed("count", "anggota"))
    print("  Garis keturunan terbesar:")
    ukuran.orderBy(F.desc("anggota")).show(12, truncate=False)

    print("  Sebaran ukuran (banyak keturunan kecil, sedikit yang besar):")
    (ukuran.withColumn("kelompok",
                       F.when(F.col("anggota") == 1, "1")
                        .when(F.col("anggota") <= 5, "2-5")
                        .when(F.col("anggota") <= 20, "6-20")
                        .when(F.col("anggota") <= 100, "21-100")
                        .otherwise(">100"))
           .groupBy("kelompok").count().orderBy("kelompok").show())



--------------------------------------------------------------------
[>] GRAPH | ukuran garis keturunan
  Garis keturunan terbesar:
+------+------------+-------+
|segmen|lineage_id  |anggota|
+------+------------+-------+
|3     |S_3_CY019832|712    |
|2     |S_2_CY009938|683    |
|5     |S_5_CY002909|668    |
|6     |S_6_AB271710|555    |
|7     |S_7_AB914511|543    |
|7     |S_7_CY002225|543    |
|2     |S_2_AB704436|510    |
|3     |S_3_CY211129|400    |
|1     |S_1_CY002479|368    |
|8     |S_8_AB434129|349    |
|8     |S_8_CY211062|336    |
|1     |S_1_OP644967|321    |
+------+------------+-------+
only showing top 12 rows

  Sebaran ukuran (banyak keturunan kecil, sedikit yang besar):
+--------+-----+
|kelompok|count|
+--------+-----+
|     2-5|  905|
|  21-100|   36|
|    6-20|  109|
|    >100|   29|
+--------+-----+

[<] OK | 0.6 detik | RAM bebas 24.5 GB


In [9]:
with Tahap("bangun jaringan reassortment", "GRAPH"):

    peta = (lineage.select(F.col("id").alias("accession"), "lineage_id", "segmen")
                   .join(fitur.select("accession", "galur", "subtipe", "negara",
                                      "wilayah", "tahun_koleksi", "inang"),
                         "accession", "inner"))
    peta.cache()
    print(f"  Sekuens terpetakan ke garis keturunan : {peta.count():,}")

    a = peta.select(F.col("galur").alias("g"), F.col("lineage_id").alias("src"),
                    F.col("segmen").alias("seg_src"))
    b = peta.select(F.col("galur").alias("g"), F.col("lineage_id").alias("dst"),
                    F.col("segmen").alias("seg_dst"))
    co_occur = (a.join(b, "g")
                 .filter(F.col("seg_src") < F.col("seg_dst"))
                 .groupBy("src", "dst", "seg_src", "seg_dst")
                 .agg(F.countDistinct("g").alias("n_galur"))
                 .filter(F.col("n_galur") >= 2))

    n_co = co_occur.count()
    print(f"  Tepi CO_OCCURS (>= 2 galur) : {n_co:,}")
    co_occur.write.mode("overwrite").parquet(jalur("graph/co_occurs"))
    co_occur.orderBy(F.desc("n_galur")).show(10, truncate=False)



--------------------------------------------------------------------
[>] GRAPH | bangun jaringan reassortment


  Sekuens terpetakan ke garis keturunan : 13,965


  Tepi CO_OCCURS (>= 2 galur) : 100
+------------+------------+-------+-------+-------+
|src         |dst         |seg_src|seg_dst|n_galur|
+------------+------------+-------+-------+-------+
|S_2_CY009938|S_3_CY019832|2      |3      |14     |
|S_3_CY019832|S_5_CY002909|3      |5      |12     |
|S_1_CY002479|S_3_CY019832|1      |3      |10     |
|S_2_AB704436|S_3_CY211129|2      |3      |9      |
|S_1_CY002479|S_5_CY002909|1      |5      |9      |
|S_2_CY009938|S_6_AB271710|2      |6      |9      |
|S_5_CY002909|S_6_AB271710|5      |6      |9      |
|S_3_CY019832|S_6_AB271710|3      |6      |9      |
|S_7_CY002225|S_8_AB434129|7      |8      |8      |
|S_6_AB271710|S_7_CY002225|6      |7      |8      |
+------------+------------+-------+-------+-------+
only showing top 10 rows

[<] OK | 8.2 detik | RAM bebas 24.8 GB


In [10]:
with Tahap("sentralitas pada jaringan reassortment", "GRAPH"):
    t_co = co_occur.select("src", "dst")

    print("  Derajat (berapa banyak garis keturunan lain yang berpasangan):")
    d = derajat(t_co)
    d.orderBy(F.desc("derajat")).show(10, truncate=False)

    print("\n  PageRank (perantara reassortment yang paling berpengaruh):")

    e2 = t_co.union(t_co.select(F.col("dst").alias("src"), F.col("src").alias("dst")))
    pr = pagerank(e2, iterasi=10)

    pr_top = pr.orderBy(F.desc("skor")).limit(20)
    pr_top.show(20, truncate=False)

    (pr.join(d, "id", "left")
       .withColumn("segmen", F.split(F.col("id"), "_")[1])
       .write.mode("overwrite").parquet(jalur("graph/metrik_sentralitas")))
    print(f"\n  Disimpan ke {jalur('graph/metrik_sentralitas')}")



--------------------------------------------------------------------
[>] GRAPH | sentralitas pada jaringan reassortment
  Derajat (berapa banyak garis keturunan lain yang berpasangan):
+------------+-------+
|id          |derajat|
+------------+-------+
|S_3_CY019832|11     |
|S_7_CY002225|10     |
|S_2_AB704436|10     |
|S_7_AB914511|10     |
|S_2_CY009938|9      |
|S_6_AB271710|8      |
|S_6_AB649900|8      |
|S_5_CY002909|8      |
|S_8_CY211062|8      |
|S_1_OP644967|7      |
+------------+-------+
only showing top 10 rows


  PageRank (perantara reassortment yang paling berpengaruh):
      53 simpul, 10 iterasi
      iterasi 1/10
      iterasi 2/10
      iterasi 3/10
      iterasi 4/10
      iterasi 5/10
      iterasi 6/10
      iterasi 7/10
      iterasi 8/10
      iterasi 9/10
      iterasi 10/10
+------------+--------------------+
|id          |skor                |
+------------+--------------------+
|S_7_MW961477|0.04764503847096156 |
|S_8_OK022587|0.03971229848891888 |
|S_3_

In [11]:
with Tahap("silang kandidat reassortant dengan graf", "GRAPH"):
    try:
        kand = spark.read.parquet(jalur("models/kandidat_reassortant"))
        ada_kand = kand.count() > 0
    except Exception:
        ada_kand = False

    if not ada_kand:
        print("  Belum ada kandidat dari notebook 05 -- lewati.")
    else:
        print(f"  Kandidat dari notebook 05 : {kand.count():,}")
        galur_kand = kand.select("galur").distinct()

        lin_kand = (peta.join(galur_kand, "galur", "inner")
                        .select("galur", "lineage_id", "segmen").distinct())
        skor = (lin_kand.join(pr.withColumnRenamed("id", "lineage_id"),
                              "lineage_id", "left")
                        .join(derajat(co_occur.select("src", "dst"))
                              .withColumnRenamed("id", "lineage_id"),
                              "lineage_id", "left"))

        print("\n  Garis keturunan milik kandidat, diurutkan menurut PageRank:")
        skor.orderBy(F.desc("skor")).show(15, truncate=False)

        rata_kand = skor.agg(F.avg("skor")).collect()[0][0] or 0
        rata_semua = pr.agg(F.avg("skor")).collect()[0][0] or 0
        print(f"\n  PageRank rata-rata garis keturunan KANDIDAT : {rata_kand:.6f}")
        print(f"  PageRank rata-rata SELURUH garis keturunan  : {rata_semua:.6f}")
        if rata_semua:
            print(f"  Rasio : {rata_kand/rata_semua:.2f}x")
            print("\n  Rasio di atas 1 berarti kandidat dari model ML memang cenderung")
            print("  menempati posisi lebih sentral di jaringan reassortment --")
            print("  dua metode terpisah menunjuk ke galur yang sama.")
        skor.write.mode("overwrite").parquet(jalur("graph/kandidat_bersilang"))



--------------------------------------------------------------------
[>] GRAPH | silang kandidat reassortant dengan graf
  Kandidat dari notebook 05 : 1,990

  Garis keturunan milik kandidat, diurutkan menurut PageRank:


+------------+------------------------------------------+------+--------------------+-------+
|lineage_id  |galur                                     |segmen|skor                |derajat|
+------------+------------------------------------------+------+--------------------+-------+
|S_2_AB704436|A/Spain/8833/2019                         |2     |0.034389561148726724|10     |
|S_2_AB704436|A/Texas/7788/2018                         |2     |0.034389561148726724|10     |
|S_2_AB704436|A/Brisbane/144/2017                       |2     |0.034389561148726724|10     |
|S_7_AB914511|A/swine/Iowa/A01202709/2011               |7     |0.03402340208839258 |10     |
|S_7_AB914511|A/swine/Indiana/17TOSU1823/2017           |7     |0.03402340208839258 |10     |
|S_7_AB914511|A/swine/Hong Kong/3125/2011               |7     |0.03402340208839258 |10     |
|S_7_AB914511|A/swine/Tornitz/IDT14598/2012             |7     |0.03402340208839258 |10     |
|S_7_AB914511|A/swine/Minnesota/SG1395/2011             |7  

[Stage 2403:================>                                    (81 + 9) / 256]

[<] OK | 5.0 detik | RAM bebas 24.8 GB


In [12]:
with Tahap("proyeksi geografis", "GRAPH"):

    lin_hash = (peta.select("accession", "lineage_id")
                    .join(fitur.select("accession", "hash_seq"), "accession")
                    .select("hash_seq", "lineage_id").distinct())

    geo = (semua_kemunculan
           .filter(F.col("negara").isNotNull() & F.col("tahun_koleksi").isNotNull())
           .join(lin_hash, "hash_seq", "inner")
           .select("negara", "wilayah", "lineage_id", "tahun_koleksi"))

    print(f"  Pengamatan geografis dipakai : {geo.count():,}")

    pertama = (geo.groupBy("negara", "lineage_id")
                  .agg(F.min("tahun_koleksi").alias("tahun_pertama")))

    x = pertama.select(F.col("negara").alias("src"), "lineage_id",
                       F.col("tahun_pertama").alias("th_src"))
    y = pertama.select(F.col("negara").alias("dst"), "lineage_id",
                       F.col("tahun_pertama").alias("th_dst"))

    berbagi = (x.join(y, "lineage_id")
                .filter(F.col("src") != F.col("dst"))
                .filter(F.col("th_src") <= F.col("th_dst"))
                .groupBy("src", "dst")
                .agg(F.count("*").alias("lineage_bersama"),
                     F.round(F.avg(F.col("th_dst") - F.col("th_src")), 2).alias("jeda_tahun"))
                .filter(F.col("lineage_bersama") >= 3))

    print(f"  Tepi SHARES_LINEAGE : {berbagi.count():,}")
    berbagi.orderBy(F.desc("lineage_bersama")).show(12, truncate=False)
    berbagi.write.mode("overwrite").parquet(jalur("graph/geo_tepi"))

    print("\n  Negara paling sentral (PageRank pada graf berarah):")
    pr_geo = pagerank(berbagi.select("src", "dst"), iterasi=12)
    pr_geo.show(15, truncate=False)
    pr_geo.write.mode("overwrite").parquet(jalur("graph/geo_sentralitas"))



--------------------------------------------------------------------
[>] GRAPH | proyeksi geografis


  Pengamatan geografis dipakai : 25,558


  Tepi SHARES_LINEAGE : 1,424


+-----------+--------------+---------------+----------+
|src        |dst           |lineage_bersama|jeda_tahun|
+-----------+--------------+---------------+----------+
|USA        |Canada        |77             |1.82      |
|Canada     |USA           |55             |0.31      |
|USA        |United Kingdom|38             |3.34      |
|USA        |Chile         |36             |3.36      |
|USA        |New Zealand   |35             |1.63      |
|USA        |Australia     |35             |1.26      |
|USA        |Japan         |33             |3.48      |
|USA        |Mexico        |31             |1.61      |
|USA        |Nicaragua     |30             |2.3       |
|USA        |China         |30             |2.33      |
|New Zealand|USA           |30             |0.73      |
|Australia  |USA           |29             |0.41      |
+-----------+--------------+---------------+----------+
only showing top 12 rows




  Negara paling sentral (PageRank pada graf berarah):


      59 simpul, 12 iterasi
      iterasi 1/12
      iterasi 2/12
      iterasi 3/12
      iterasi 4/12
      iterasi 5/12
      iterasi 6/12
      iterasi 7/12
      iterasi 8/12
      iterasi 9/12
      iterasi 10/12
      iterasi 11/12
      iterasi 12/12
+--------------+--------------------+
|id            |skor                |
+--------------+--------------------+
|France        |0.03277608973788201 |
|Germany       |0.032771394435561874|
|New Zealand   |0.02642389440574766 |
|South Korea   |0.024884635348939442|
|Chile         |0.02454821878444926 |
|Russia        |0.024258405618064843|
|Kenya         |0.02407070150678059 |
|Italy         |0.022110351768160223|
|Japan         |0.02108902486184876 |
|Switzerland   |0.020656533826348453|
|China         |0.019846935143829982|
|Canada        |0.019813200223041928|
|United Kingdom|0.018808188016416552|
|Turkey        |0.018512597411422096|
|South Africa  |0.01809563973670705 |
+--------------+--------------------+
only showing top 15

In [ ]:
with Tahap("hubungkan ke Neo4j dan siapkan skema", "GRAPH-DB"):
    from neo4j import GraphDatabase

    NEO4J_URI = os.environ.get("BDA_NEO4J", "bolt://neo4j:7687")
    NEO4J_AUTH = neo4j_auth()

    driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
    driver.verify_connectivity()

    with driver.session() as ses:
        v = ses.run("CALL dbms.components() YIELD name, versions "
                    "RETURN name, versions[0] AS versi").single()
        print(f"  {v['name']} {v['versi']}  di {NEO4J_URI}")

        for c in [
            "CREATE CONSTRAINT lineage_id IF NOT EXISTS "
            "FOR (n:GarisKeturunan) REQUIRE n.id IS UNIQUE",
            "CREATE CONSTRAINT negara_nama IF NOT EXISTS "
            "FOR (n:Negara) REQUIRE n.nama IS UNIQUE",
        ]:
            ses.run(c)
        print("  constraint unik dipasang untuk :GarisKeturunan dan :Negara")

        n_lama = ses.run("MATCH (n) RETURN count(n) AS n").single()["n"]
        if n_lama:
            ses.run("MATCH (n) DETACH DELETE n")
            print(f"  {n_lama:,} simpul lama dihapus")

def muat_neo4j(cypher, baris, ukuran_bets=5_000, label=""):
    total = 0
    with driver.session() as ses:
        for i in range(0, len(baris), ukuran_bets):
            bets = baris[i:i + ukuran_bets]
            ses.run(cypher, baris=bets)
            total += len(bets)
    if label:
        print(f"  {label:<38} {total:>8,}")
    return total


In [14]:
with Tahap("muat jaringan reassortment dan geografis ke Neo4j", "GRAPH-DB"):

    MAKS_TEPI_REASSORT = 200_000
    MAKS_TEPI_GEO = 50_000

    lin = (peta.select("lineage_id", "segmen").distinct()
               .withColumn("gen", F.element_at(
                   F.array(*[F.lit(SEGMEN[s][0]) for s in range(1, 9)]),
                   F.col("segmen").cast("int")))
               .toPandas())
    muat_neo4j(
        "UNWIND $baris AS b "
        "MERGE (n:GarisKeturunan {id: b.lineage_id}) "
        "SET n.segmen = b.segmen, n.gen = b.gen",
        lin.to_dict("records"), label="simpul :GarisKeturunan")

    ko = (co_occur.orderBy(F.desc("n_galur")).limit(MAKS_TEPI_REASSORT)
                  .toPandas())
    muat_neo4j(
        "UNWIND $baris AS b "
        "MATCH (a:GarisKeturunan {id: b.src}) "
        "MATCH (c:GarisKeturunan {id: b.dst}) "
        "MERGE (a)-[r:BERPASANGAN]->(c) "
        "SET r.n_galur = b.n_galur",
        ko.to_dict("records"), label="tepi :BERPASANGAN")

    geo_tepi = (berbagi.orderBy(F.desc("lineage_bersama"))
                       .limit(MAKS_TEPI_GEO).toPandas())

    negara = sorted(set(geo_tepi["src"]) | set(geo_tepi["dst"]))
    muat_neo4j(
        "UNWIND $baris AS b MERGE (n:Negara {nama: b.nama})",
        [{"nama": x} for x in negara], label="simpul :Negara")

    muat_neo4j(
        "UNWIND $baris AS b "
        "MATCH (a:Negara {nama: b.src}) "
        "MATCH (c:Negara {nama: b.dst}) "
        "MERGE (a)-[r:MENDAHULUI]->(c) "
        "SET r.lineage_bersama = b.lineage_bersama, r.jeda_tahun = b.jeda_tahun",
        geo_tepi.to_dict("records"), label="tepi :MENDAHULUI")

    with driver.session() as ses:
        ringkas = ses.run(
            "MATCH (n) WITH labels(n)[0] AS label, count(*) AS n "
            "RETURN label, n ORDER BY n DESC").data()
        tepi_db = ses.run(
            "MATCH ()-[r]->() WITH type(r) AS jenis, count(*) AS n "
            "RETURN jenis, n ORDER BY n DESC").data()
    print()
    display(pd.DataFrame(ringkas))
    display(pd.DataFrame(tepi_db))



--------------------------------------------------------------------
[>] GRAPH-DB | muat jaringan reassortment dan geografis ke Neo4j


  simpul :GarisKeturunan                    1,079
  tepi :BERPASANGAN                           100


  simpul :Negara                               59
  tepi :MENDAHULUI                          1,424



,label,n
0,GarisKeturunan,1079
1,Negara,59


,jenis,n
0,MENDAHULUI,1424
1,BERPASANGAN,100


[<] OK | 15.7 detik | RAM bebas 25.0 GB


In [15]:
with Tahap("analitik graf di dalam Neo4j (Cypher + GDS)", "GRAPH-DB"):
    def cypher(q, **p):
        with driver.session() as ses:
            return pd.DataFrame(ses.run(q, **p).data())

    print("  Garis keturunan dengan pasangan terbanyak:")
    display(cypher("""
        MATCH (n:GarisKeturunan)-[r:BERPASANGAN]-()
        RETURN n.id AS garis_keturunan, n.gen AS gen,
               count(r) AS derajat, sum(r.n_galur) AS total_galur
        ORDER BY derajat DESC LIMIT 10
    """))

    ada_gds = not cypher(
        "SHOW PROCEDURES YIELD name "
        "WHERE name = 'gds.pageRank.stream' RETURN name").empty

    if not ada_gds:
        print("\n  [!] Graph Data Science tidak tersedia -- PageRank dilewati.")
        print("      Plugin diunduh saat container neo4j pertama kali naik;")
        print("      periksa koneksi internet lalu buat ulang containernya:")
        print("        docker compose up -d --force-recreate neo4j")
    else:

        with driver.session() as ses:
            ses.run("CALL gds.graph.drop('reassort', false) YIELD graphName")
            ses.run("""
                CALL gds.graph.project('reassort', 'GarisKeturunan',
                     {BERPASANGAN: {orientation: 'UNDIRECTED',
                                    properties: 'n_galur'}})
            """)

        print("\n  PageRank -- perantara reassortment paling berpengaruh:")
        pr_neo = cypher("""
            CALL gds.pageRank.stream('reassort',
                 {relationshipWeightProperty: 'n_galur'})
            YIELD nodeId, score
            RETURN gds.util.asNode(nodeId).id  AS garis_keturunan,
                   gds.util.asNode(nodeId).gen AS gen,
                   round(score, 5) AS pagerank
            ORDER BY pagerank DESC LIMIT 15
        """)
        display(pr_neo)

        print("\n  Louvain -- konstelasi genom yang cenderung beredar bersama:")
        display(cypher("""
            CALL gds.louvain.stream('reassort')
            YIELD nodeId, communityId
            WITH communityId,
                 count(*) AS anggota,
                 collect(gds.util.asNode(nodeId).gen)[0..8] AS contoh_gen
            RETURN communityId AS komunitas, anggota, contoh_gen
            ORDER BY anggota DESC LIMIT 10
        """))

        pr_spark = (pr.orderBy(F.desc("skor")).limit(15)
                      .toPandas().rename(columns={"id": "garis_keturunan"}))
        irisan = set(pr_neo["garis_keturunan"]) & set(pr_spark["garis_keturunan"])
        print(f"\n  Irisan 15 teratas Spark dan Neo4j : {len(irisan)} dari 15")
        print("  Yang penting peringkatnya, bukan skor mutlaknya. Kalau dua")
        print("  mesin yang sepenuhnya terpisah menyepakati simpul mana yang")
        print("  paling sentral, temuannya jauh lebih sulit dibantah.")



--------------------------------------------------------------------
[>] GRAPH-DB | analitik graf di dalam Neo4j (Cypher + GDS)
  Garis keturunan dengan pasangan terbanyak:


,garis_keturunan,gen,derajat,total_galur
0,S_3_CY019832,PA,11,68
1,S_7_CY002225,M,10,40
2,S_7_AB914511,M,10,40
3,S_2_AB704436,PB1,10,38
4,S_2_CY009938,PB1,9,56
5,S_6_AB271710,NA,8,53
6,S_5_CY002909,NP,8,54
7,S_6_AB649900,NA,8,20
8,S_8_CY211062,NS,8,29
9,S_3_CY211129,PA,7,28



  PageRank -- perantara reassortment paling berpengaruh:


,garis_keturunan,gen,pagerank
0,S_7_MW961477,M,2.65636
1,S_3_CY019832,PA,2.26686
2,S_7_AB914511,M,2.12678
3,S_2_AB704436,PB1,2.01808
4,S_8_OK022587,NS,1.96091
5,S_2_CY009938,PB1,1.92835
6,S_5_CY002909,NP,1.70551
7,S_6_AB271710,NA,1.69040
8,S_8_CY211062,NS,1.62228
9,S_3_CY211129,PA,1.49347



  Louvain -- konstelasi genom yang cenderung beredar bersama:


,komunitas,anggota,contoh_gen
0,71,17,"[HA, NS, PB1, NA, NP, PA, M, PA]"
1,28,15,"[NS, M, NA, NP, PB2, PB2, HA, PB1]"
2,56,11,"[NS, PB2, PA, M, PB1, NP, HA, HA]"
3,130,2,"[PB1, M]"
4,714,2,"[M, PA]"
5,385,2,"[NS, PA]"
6,176,2,"[PB2, NP]"
7,482,2,"[M, PB1]"
8,11,1,[PA]
9,10,1,[HA]



  Irisan 15 teratas Spark dan Neo4j : 13 dari 15
  Yang penting peringkatnya, bukan skor mutlaknya. Kalau dua
  mesin yang sepenuhnya terpisah menyepakati simpul mana yang
  paling sentral, temuannya jauh lebih sulit dibantah.
[<] OK | 1.7 detik | RAM bebas 25.0 GB


In [16]:
with Tahap("kueri lintasan -- yang sukar dinyatakan di Spark", "GRAPH-DB"):

    print("  1. Jalur penyebaran terpendek antar negara")
    print("     'Lewat negara mana garis keturunan berpindah dari A ke B?'")
    display(cypher("""
        MATCH (a:Negara)-[:MENDAHULUI*1..3]->(b:Negara)
        WHERE a.nama <> b.nama
        WITH a, b, count(*) AS jalur
        RETURN a.nama AS dari, b.nama AS ke, jalur
        ORDER BY jalur DESC LIMIT 12
    """))

    print("\n  2. Negara paling sentral sebagai SUMBER")
    print("     Derajat keluar tinggi berarti sering mendeteksi lebih dulu.")
    display(cypher("""
        MATCH (n:Negara)
        OPTIONAL MATCH (n)-[keluar:MENDAHULUI]->()
        OPTIONAL MATCH (n)<-[masuk:MENDAHULUI]-()
        WITH n, count(DISTINCT keluar) AS mendahului,
                count(DISTINCT masuk)  AS didahului
        WHERE mendahului + didahului > 0
        RETURN n.nama AS negara, mendahului, didahului,
               mendahului - didahului AS selisih
        ORDER BY selisih DESC LIMIT 12
    """))

    print("\n  3. Garis keturunan yang menjembatani DUA gen berbeda")
    print("     Pola inilah tanda reassortment: satu garis keturunan segmen")
    print("     yang berpasangan dengan garis keturunan dari gen lain.")
    display(cypher("""
        MATCH (a:GarisKeturunan)-[r:BERPASANGAN]-(b:GarisKeturunan)
        WHERE a.gen <> b.gen
        WITH a, count(DISTINCT b.gen) AS gen_pasangan, sum(r.n_galur) AS galur
        WHERE gen_pasangan >= 2
        RETURN a.id AS garis_keturunan, a.gen AS gen,
               gen_pasangan, galur
        ORDER BY gen_pasangan DESC, galur DESC LIMIT 12
    """))

    print("\n  Bandingkan dengan ekuivalennya di Spark: kueri pertama saja")
    print("  menuntut tiga self-join berantai atas tabel tepi, dan kedalaman")
    print("  maksimumnya harus ditulis tangan. Di Cypher itu satu pola,")
    print("  '*1..3', dan mesin yang mengurus penelusurannya.")



--------------------------------------------------------------------
[>] GRAPH-DB | kueri lintasan -- yang sukar dinyatakan di Spark
  1. Jalur penyebaran terpendek antar negara
     'Lewat negara mana garis keturunan berpindah dari A ke B?'


,dari,ke,jalur
0,USA,France,1274
1,Australia,France,1262
2,China,France,1261
3,Mexico,France,1253
4,USA,Kenya,1247
5,Hong Kong,France,1247
6,Thailand,France,1242
7,China,Kenya,1237
8,Australia,Kenya,1235
9,USA,Germany,1234



  2. Negara paling sentral sebagai SUMBER
     Derajat keluar tinggi berarti sering mendeteksi lebih dulu.


,negara,mendahului,didahului,selisih
0,USA,56,35,21
1,Australia,50,33,17
2,Hong Kong,46,30,16
3,Mexico,49,34,15
4,Taiwan,46,31,15
5,China,52,38,14
6,Netherlands,34,21,13
7,Argentina,36,23,13
8,Iran,27,15,12
9,Thailand,45,33,12



  3. Garis keturunan yang menjembatani DUA gen berbeda
     Pola inilah tanda reassortment: satu garis keturunan segmen
     yang berpasangan dengan garis keturunan dari gen lain.


,garis_keturunan,gen,gen_pasangan,galur
0,S_3_CY019832,PA,7,68
1,S_2_CY009938,PB1,7,56
2,S_7_AB914511,M,7,40
3,S_7_CY002225,M,7,40
4,S_2_AB704436,PB1,7,38
5,S_8_CY211062,NS,7,29
6,S_1_OP644967,PB2,7,23
7,S_5_CY002909,NP,6,54
8,S_6_AB271710,NA,6,53
9,S_3_CY211129,PA,6,28



  Bandingkan dengan ekuivalennya di Spark: kueri pertama saja
  menuntut tiga self-join berantai atas tabel tepi, dan kedalaman
  maksimumnya harus ditulis tangan. Di Cypher itu satu pola,
  '*1..3', dan mesin yang mengurus penelusurannya.
[<] OK | 1.2 detik | RAM bebas 24.9 GB


In [17]:
with Tahap("sintesis graph analytics", "GRAPH"):
    ringkas_graf = {
        "run_id": RUN_ID,
        "k": K,
        "ambang_jarak_jaccard": AMBANG_JARAK,
        "tepi_kemiripan": int(tepi.count()),
        "garis_keturunan": int(lineage.select("lineage_id").distinct().count()),
        "tepi_co_occurs": int(co_occur.count()),
        "tepi_geografis": int(berbagi.count()),
        "implementasi_graf": "Spark DataFrame asli: label propagation + PageRank",
        "per_segmen": df_tepi_ringkas.to_dict("records"),
    }
    (BASE / "graph").mkdir(parents=True, exist_ok=True)
    (BASE / "graph" / "ringkasan_graf.json").write_text(
        json.dumps(ringkas_graf, indent=2, default=str), encoding="utf-8")
    print(json.dumps({k: v for k, v in ringkas_graf.items() if k != "per_segmen"},
                     indent=2))



--------------------------------------------------------------------
[>] GRAPH | sintesis graph analytics


{
  "run_id": "run_20261004T162404Z",
  "k": 8,
  "ambang_jarak_jaccard": 0.12,
  "tepi_kemiripan": 358212,
  "garis_keturunan": 1079,
  "tepi_co_occurs": 100,
  "tepi_geografis": 1424,
  "implementasi_graf": "Spark DataFrame asli: label propagation + PageRank"
}
[<] OK | 11.7 detik | RAM bebas 24.9 GB


In [18]:
with Tahap("visualisasi graf", "MONITORING"):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import networkx as nx

    top = [r["id"] for r in pr.orderBy(F.desc("skor")).limit(60).collect()]
    sub = (co_occur.filter(F.col("src").isin(top) & F.col("dst").isin(top))
                   .select("src", "dst", "n_galur").toPandas())

    if len(sub):
        G = nx.Graph()
        for _, r in sub.iterrows():
            G.add_edge(r["src"], r["dst"], weight=int(r["n_galur"]))
        fig, ax = plt.subplots(figsize=(12, 9))
        pos = nx.spring_layout(G, seed=CFG["seed"], k=0.6)
        bobot = [G[u][v]["weight"] for u, v in G.edges()]
        maks = max(bobot) if bobot else 1
        nx.draw_networkx_edges(G, pos, width=[0.4 + 3*w/maks for w in bobot],
                               alpha=0.35, ax=ax)
        ukuran_simpul = [120 + 40 * G.degree(nd) for nd in G.nodes()]
        warna = [int(str(nd).split("_")[1]) for nd in G.nodes()]
        nx.draw_networkx_nodes(G, pos, node_size=ukuran_simpul, node_color=warna,
                               cmap="tab10", alpha=0.9, ax=ax)
        ax.set_title("Jaringan reassortment — 60 garis keturunan segmen paling sentral\n"
                     "warna = nomor segmen, tebal tepi = jumlah galur yang berbagi",
                     fontsize=12)
        ax.axis("off")
        plt.tight_layout()
        berkas = OUT / "graf_reassortment.png"
        plt.savefig(berkas, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"  Gambar disimpan: {berkas}")
        print(f"  Simpul digambar: {G.number_of_nodes()}, tepi: {G.number_of_edges()}")
    else:
        print("  Sub-graf kosong -- turunkan ambang atau perbesar data.")



--------------------------------------------------------------------
[>] MONITORING | visualisasi graf
  Gambar disimpan: /workspace/output/graf_reassortment.png
  Simpul digambar: 53, tepi: 100
[<] OK | 2.2 detik | RAM bebas 24.9 GB


In [19]:
try:
    driver.close()
    print("Koneksi Neo4j ditutup.")
except NameError:
    pass

display(ringkas_zona())
display(jejak_df())
stop_spark()
print("\nSelesai. Lanjut ke 07_mart_dashboard.ipynb")


Koneksi Neo4j ditutup.


,zona,isi,ukuran
0,lake/fasta,152,162.2 MB
1,lake/meta_csv,49,367.6 MB
2,stage,1,1.3 KB
3,features,1,735.0 B
4,models,4,2.2 KB
5,graph,1,1.5 KB
6,mart,0,0.0 B
7,output,4,464.8 KB


,run_id,lapisan,tahap,status,detik,ram_delta_gb,ram_bebas_gb,waktu
0,run_20261004T162404Z,GRAPH,muat graf kemiripan yang sudah tersimpan,OK,2.43,0.11,31.9,2026-10-04T16:25:25.148815+00:00
1,run_20261004T162404Z,GRAPH,uji satu segmen sebelum menjalankan semuanya,OK,61.89,2.69,29.3,2026-10-04T16:26:27.050038+00:00
2,run_20261004T162404Z,GRAPH,bangun graf kemiripan untuk segmen sisanya,OK,1101.64,4.14,25.1,2026-10-04T16:44:48.700534+00:00
3,run_20261004T162404Z,GRAPH,komponen terhubung -> garis keturunan segmen,OK,50.58,0.90,24.2,2026-10-04T16:45:39.290718+00:00
4,run_20261004T162404Z,GRAPH,ukuran garis keturunan,OK,0.60,-0.28,24.5,2026-10-04T16:45:39.904510+00:00
5,run_20261004T162404Z,GRAPH,bangun jaringan reassortment,OK,8.20,-0.35,24.8,2026-10-04T16:45:48.108998+00:00
6,run_20261004T162404Z,GRAPH,sentralitas pada jaringan reassortment,OK,7.32,-0.17,25.0,2026-10-04T16:45:55.438516+00:00
7,run_20261004T162404Z,GRAPH,silang kandidat reassortant dengan graf,OK,5.05,0.22,24.8,2026-10-04T16:46:00.494485+00:00
8,run_20261004T162404Z,GRAPH,proyeksi geografis,OK,52.37,-0.69,25.4,2026-10-04T16:46:52.879120+00:00
9,run_20261004T162404Z,GRAPH-DB,hubungkan ke Neo4j dan siapkan skema,OK,2.20,0.09,25.4,2026-10-04T16:46:55.092022+00:00


Spark dihentikan. Proses Java tersisa: 8

Selesai. Lanjut ke 07_mart_dashboard.ipynb
